In [ ]:
#TASK1
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import time
import os
import duckdb
# Step 1: Create Dataset
np.random.seed(42)
num_rows = 500_000
cities = ["Berlin", "Tokyo", "New York", "London", "Paris", "Seoul", "Sydney", "Mumbai", "Toronto", "Madrid"]

df = pd.DataFrame({
    'user_id': np.arange(1, num_rows + 1),
    'city': np.random.choice(cities, num_rows),
    'score': np.random.uniform(0, 100, num_rows),
    'active': np.random.choice([True, False], num_rows),
    'signup_date': pd.to_datetime('2023-01-01') + pd.to_timedelta(np.random.randint(0, 1095, num_rows), unit='D'),
    'age': np.random.randint(18, 81, num_rows),
    'sessions': np.random.randint(0, 501, num_rows),
    'revenue': np.random.uniform(0, 1000, num_rows)
})

# Step 2: Save and Inspect Parquet
file_parquet = 'data.parquet'
df.to_parquet(file_parquet, index=False)
parquet_file = pq.ParquetFile(file_parquet)

print(f"Row Groups: {parquet_file.num_row_groups}")
print(f"Rows: {parquet_file.metadata.num_rows}, Cols: {parquet_file.metadata.num_columns}")
print(f"Schema: \n{parquet_file.schema}")

# Inspect first row group columns
first_rg = parquet_file.metadata.row_group(0)
for i in range(first_rg.num_columns):
    col = first_rg.column(i)
    print(f"Col: {col.path_in_schema} | Type: {col.physical_type} | Size: {col.total_compressed_size} bytes")

# Step 3: Compare with CSV
file_csv = 'data.csv'
df.to_csv(file_csv, index=False)
csv_size = os.path.getsize(file_csv) / 1024
pq_size = os.path.getsize(file_parquet) / 1024

print(f"CSV Size: {csv_size:.2f} KB | Parquet Size: {pq_size:.2f} KB")
print(f"Compression Ratio: {csv_size/pq_size:.2f}x")

## STEP 4
Question : Write a markdown cell explaining what information the Parquet metadata provides that CSV does not, and why that matters for query performance.

Answer : Parquet stores schema and statistics (min/max/nulls) in the file footer. Unlike CSV, which is just a stream of text, Parquet allows the query engine to skip entire chunks of data if the metadata shows the requested values aren't in that range.

In [ ]:
#TASK2

#Step1
start = time.time()
_ = pd.read_parquet(file_parquet)
print(f"Full Parquet Read: {time.time() - start:.4f}s")

#Step2
start = time.time()
_ = pd.read_parquet(file_parquet, columns=['city', 'revenue'])
print(f"Selective Parquet Read: {time.time() - start:.4f}s")

#Step 3
start_csv_selective = time.time()

df_csv_small = pd.read_csv('data.csv', usecols=['city', 'revenue'])

end_csv_selective = time.time()
print(f"CSV Selective Read Time: {end_csv_selective - start_csv_selective:.4f} s")



## Step 4
Question : Write a markdown cell explaining why Parquet selective reads are faster. Connect your explanation to the column-chunk layout you observed in Task 1.

Answer :
CSV (row-based): Data is stored row by row. To access a single column (e.g., city), the system still has to scan through entire rows, reading or skipping unnecessary fields like user_id, age, score, etc.
Parquet (columnar): Data is stored in column chunks. If you only need city and revenue, the reader directly loads only those columns and completely skips the rest.

In [ ]:
# TASK 3
# Step 1: PyArrow
dataset = ds.dataset('data.parquet', format="parquet")
start_pa = time.time()
table_filtered = dataset.to_table(filter=ds.field("age") > 50)
df_pa_filtered = table_filtered.to_pandas()
end_pa = time.time()

pa_time = end_pa - start_pa
pa_rows = len(df_pa_filtered)

# Step 2: Pandas 
start_pd = time.time()
df_full = pd.read_parquet('data.parquet')
df_pd_filtered = df_full[df_full['age'] > 50]
end_pd = time.time()

pd_time = end_pd - start_pd
pd_rows = len(df_pd_filtered)

# Step 4: DuckDB SQL Approach
start_db = time.time()
duckdb_result = duckdb.sql("SELECT * FROM 'data.parquet' WHERE age > 50").df()
end_db = time.time()

db_time = end_db - start_db

# Results Comparison
print(f"PyArrow Pushdown: {pa_time:.4f}s | Rows: {pa_rows}")
print(f"Pandas Eager:    {pd_time:.4f}s | Rows: {pd_rows}")
print(f"DuckDB SQL:      {db_time:.4f}s | Rows: {len(duckdb_result)}")

## STEP 3
I/O Reduction:
Parquet files store statistics (min and max values) for each row group. For example, if a row group has max(age) = 45 and your query is looking for values greater than 50, the system can skip that entire block without reading it from disk. This is known as predicate pushdown and significantly reduces unnecessary I/O.

Memory Efficiency:
In a typical pandas workflow, the entire dataset is loaded into RAM and then filtered. With Parquet’s predicate pushdown, only the rows that satisfy the filter condition are loaded into memory. This is crucial when working with datasets that are close to or exceed available memory capacity.

CPU Efficiency:
Since less data is read and processed, the amount of deserialization (converting data from disk format into memory objects) is reduced. This leads to a significant decrease in CPU usage, making queries faster and more efficient overall.

In [ ]:
#TASK4
con = duckdb.connect()

def time_query(func, name):
    start = time.time()
    result = func()
    end = time.time()
    duration = end - start
    print(f"{name} execution time: {duration:.4f} seconds")
    return duration

# Query 1: Count records per city 
def q1_pandas(): return df.groupby('city').size()
def q1_duckdb(): return duckdb.sql("SELECT city, COUNT(*) FROM df GROUP BY city").df()

# Query 2: Average score by city, ordered DESC 
def q2_pandas(): return df.groupby('city')['score'].mean().sort_values(ascending=False)
def q2_duckdb(): return duckdb.sql("SELECT city, AVG(score) as avg_score FROM df GROUP BY city ORDER BY avg_score DESC").df()

# Query 3: Percentage of active users with score > 75 
def q3_pandas():
    # Filter active users with score > 75 and group by city
    subset = df[(df['active'] == True) & (df['score'] > 75)]
    counts = subset.groupby('city').size()
    total = df.groupby('city').size()
    return (counts / total) * 100

def q3_duckdb():
    return duckdb.sql("""
        SELECT city, 
        (COUNT(CASE WHEN active = True AND score > 75 THEN 1 END) * 100.0 / COUNT(*)) as percentage
        FROM df GROUP BY city
    """).df()

# Query 4: Top 10 users by score per city (Window Function)
def q4_pandas():
    return df.sort_values(['city', 'score'], ascending=[True, False]).groupby('city').head(10)

def q4_duckdb():
    return duckdb.sql("""
        SELECT * FROM (
            SELECT *, RANK() OVER (PARTITION BY city ORDER BY score DESC) as rnk
            FROM df
        ) WHERE rnk <= 10
    """).df()

# Query 5: Running total of scores by user_id, partitioned by city
def q5_pandas():
    # In pandas, this operation is done using transform/cumsum
    df_sorted = df.sort_values(['city', 'user_id'])
    return df_sorted.groupby('city')['score'].cumsum()

def q5_duckdb():
    return duckdb.sql("""
        SELECT city, user_id, score,
        SUM(score) OVER (PARTITION BY city ORDER BY user_id) as running_total
        FROM df
    """).df()

# Execution and Comparison
print("Query 1")
p1 = time_query(q1_pandas, "Pandas Q1")
d1 = time_query(q1_duckdb, "DuckDB Q1")

print("\nQuery 2:")
p2 = time_query(q2_pandas, "Pandas Q2")
d2 = time_query(q2_duckdb, "DuckDB Q2")

print("\nQuery 3:")
p3 = time_query(q3_pandas, "Pandas Q3")
d3 = time_query(q3_duckdb, "DuckDB Q3")

print("\nQuery 4")
p4 = time_query(q4_pandas, "Pandas Q4")
d4 = time_query(q4_duckdb, "DuckDB Q4")

print("\nQuery 5")
p5 = time_query(q5_pandas, "Pandas Q5")
d5 = time_query(q5_duckdb, "DuckDB Q5")

## TASK4 - Step5
1. Which tool was easier to express each query in?
Simple Aggregations (Queries 1 & 2): pandas feels more natural for Python developers. Methods like .groupby().size() or .mean() are very concise and readable in a Jupyter environment.

Complex Analytics (Queries 3, 4, & 5): DuckDB (SQL) wins here. SQL was specifically built for relational algebra. Expressing a Window Function (like RANK() or SUM() OVER) in SQL is far more readable than the pandas equivalent, which requires multiple steps like sort_values(), groupby(), and transform() or apply().

2. Which was faster?

DuckDB.

In almost every execution, DuckDB outperformed pandas. This is because DuckDB uses Vectorized Query Execution. While pandas often processes data using high-level Python objects or iterative logic, DuckDB processes data in large "vectors" (batches) of values using highly optimized C++ code that stays within the CPU cache.

3. For which queries did the difference matter most?
The difference was most significant in Query 4 (Top 10 users) and Query 5 (Running Total).

Why? These queries involve heavy sorting and partitioning.

pandas has to manage memory for intermediate objects and often relies on slower "apply" or "transform" logic that doesn't scale as well.

DuckDB optimizes the "Execution Plan" before running the query. It knows exactly how to partition the data and calculate the running total in a single pass over the data blocks.

In [ ]:
#TASK5

# Step 1: Convert pandas DataFrame to Arrow Table
arrow_table = pa.Table.from_pandas(df)
print("Step 1: Conversion to Arrow Table successful.")

# Step 2: Inspect Schema vs pandas dtypes
print("\nStep 2: Schema Comparison")
print("Pandas Dtypes:\n", df.dtypes.head(3))

print("\nArrow Schema (First 3 fields):")
for i in range(3):
    print(arrow_table.schema[i])

# Step 3: Write Arrow Table to Parquet and read it back
pq.write_table(arrow_table, 'arrow_bridge.parquet')
new_arrow_table = pq.read_table('arrow_bridge.parquet')
print("\nStep 3: Parquet write/read via Arrow successful.")

# Step 4: Convert back to pandas and verify
df_back = new_arrow_table.to_pandas()
verification = df.equals(df_back)
print(f"\nStep 4: Data matches original? {verification}")

# Step 5: Demonstrate dtype_backend="pyarrow"
# This is a modern pandas feature that keeps data in Arrow format inside pandas
df_arrow_backend = pd.read_parquet('arrow_bridge.parquet', dtype_backend="pyarrow")

print("\nStep 5: Dtype Backend Comparison")
print("Traditional Pandas Dtype (city):", df['city'].dtype)
print("Arrow-backed Pandas Dtype (city):", df_arrow_backend['city'].dtype)

## STEP6
The Role of Apache Arrow in the Modern Analytics Stack
Apache Arrow is the in-memory standard that allows the different tools in this lab to speak the same language. Without Arrow, moving data between a Parquet file, a SQL engine (DuckDB), and a Python library (pandas) would be slow and memory-intensive.

1. Connecting Parquet to the Stack (Disk to RAM)
Parquet is a columnar storage format optimized for sitting on your hard drive (compressed and efficient). Arrow is a columnar memory format optimized for your CPU. Because both use a columnar structure, reading Parquet into Arrow is nearly a direct copy. There is no need to "rearrange" the data, which makes loading incredibly fast.

2. Connecting DuckDB to pandas (The Bridge)
In Task 4, you queried a pandas DataFrame using DuckDB SQL. In the past, this would require "copying" the data into the database. Thanks to Arrow:

Zero-Copy Sharing: DuckDB can look at the Arrow-backed memory of a pandas DataFrame and run SQL on it directly.

No Overhead: There is no "tax" or waiting time to convert data between these tools.

3. Modern pandas (The pyarrow Backend)
Traditionally, pandas used NumPy as its backbone, which often struggled with strings and null values. By using dtype_backend="pyarrow" (as seen in Task 5), pandas adopts Arrow's memory format. This results in:

Lower Memory Usage: Arrow is more efficient at storing strings and booleans.

Better Null Handling: Arrow has a built-in way to handle "missing data" that is more consistent than traditional pandas.